# Demo — Auditing a Regulated Number to Source

A CSRD disclosure is only as trustworthy as its traceability to source. This walkthrough runs
the reporting pipeline, compares the disclosed quarterly figures against recomputed figures in
`rep.reconciliation`, and demonstrates how to drill down from a quarterly summary into the
underlying order and shipment records in `rep.per_order`. The exercise asks you to trace a
quarter to source, isolate the anomalous rows driving the discrepancy, and write the
traceability assessment.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
rep = gc.run_sustainability_report()
display(rep.quarterly)
rep.reconciliation

## Drill down from a quarterly figure to `rep.per_order`
Notice in `rep.reconciliation` that **every quarter fails reconciliation** (`matches == False`):
`disclosed_carbon_kg` (which sums `shipment.carbon_kg` as-is) is 8x–65x larger than
`recomputed_carbon_kg`.

Use `rep.per_order` to inspect the order-and-shipment grain and compare the distribution of
`carbon_kg` across shipments:

In [ ]:
print("Total shipped order rows:", len(rep.per_order))
print("\nDistribution of per-shipment carbon_kg across all quarters:")
print(rep.per_order["carbon_kg"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2))
rep.per_order.head()

**Read the distribution:** up through the 95th percentile (`9.41 kg`), per-shipment emissions
sit in a normal single-digit kilogram range (`0.4`–`9.5 kg`), yet the 99th percentile and `max`
jump into the **thousands** (`9484.0 kg`), inflating `disclosed_carbon_kg` every quarter.

**Takeaway / your turn:** in the exercise you'll filter `rep.per_order` on a specific quarter,
isolate the exact rows driving `delta_kg`, explain the root cause of the 1000x jump, and write
the traceability assessment.